# Nordgren PINNs to VQE: Advancing Hydraulic Fracturing Simulations in Shale Reservoirs

**Paper:** Wayo, D.D.K., Irawan, S., Noor, M.Z.B.M., Zafar, M., Juziyeva, S., Saporetti, C.M., Goliatt, L., Hazlett, R. (2026). *Nordgren PINNs to VQE: Advancing Hydraulic Fracturing Simulations in Shale Reservoirs.* International Journal for Numerical and Analytical Methods in Geomechanics.

**Carpeta origen:** `PINNs/1. mecanica de fluidos/Nordgren_PINNs_to_VQE_Advancing_Hydraulic_Fracturi.pdf`

## Como se usan las PINNs en este paper

El paper usa una PINN para resolver la **ecuacion de Nordgren**, que gobierna la evolucion del ancho de una fractura hidraulica $w_0(x,t)$ en un yacimiento de lutita (Eq. 1):

$$\frac{E'}{128\mu H}\frac{\partial^2}{\partial x^2}(w_0^2) = \frac{8C_L}{\pi\sqrt{t-t_0}} + \frac{\partial w_0}{\partial t}$$

donde $E'$ es el modulo elastico efectivo, $\mu$ la viscosidad del fluido, $H$ la altura de la fractura y $C_L$ el coeficiente de perdida por filtracion (*leak-off*). La red $w_{PINN}(x,t;\theta)$ (4 capas ocultas x 256 neuronas, tanh, Seccion 3.2.1) se entrena minimizando (Eq. 2-4):

$$\mathcal{L}=\lambda_1\mathcal{L}_{physics}+\lambda_2\mathcal{L}_{boundary},\qquad \mathcal{L}_{physics}=\frac{1}{N}\sum_i\Big[\frac{E'}{128\mu H}\partial_{xx}(w_{PINN}^2)-\frac{8C_L}{\pi\sqrt{t-t_0}}-\partial_t w_{PINN}\Big]^2$$

con condiciones de contorno $w_0=0$ en las puntas de la fractura ($x=0$ y $x=1$, Seccion 2.1.1: "$N_b=2$ boundary points at $x=\{0,1\}$ with $w=0$", peso $\lambda_{bc}=30$) mas la condicion inicial $w_0(x,0)=0$. El entrenamiento usa una estrategia de optimizadores por etapas: **Adam** (1000 epocas, *learning rate* con decaimiento exponencial) seguido de **L-BFGS** (y opcionalmente Newton-CG, que aqui no se reproduce por no ser estandar en PyTorch).

El paper reporta honestamente (Seccion 5.1.1) que la PINN **subestima sistematicamente** el crecimiento cuadratico de la solucion analitica, atribuido al desbalance de escala del termino $w_0^2$, la escasa supervision de frontera, y la falta de una restriccion de no negatividad — el Algoritmo 1 del paper sugiere opcionalmente aplicar `softplus` a la salida para forzar $w_0\geq 0$. Este cuaderno reproduce fielmente la arquitectura, la funcion de perdida, las condiciones de contorno/iniciales, la estrategia de optimizador por etapas, **e incluye la correccion opcional de no-negatividad** que el propio paper identifica como mejora natural.

## Repositorio publico de referencia

El PDF no incluye un repositorio de codigo propio (se menciona el uso de TensorFlow en Google Colab y Qiskit en Qbraid, pero sin enlace publico). Como referencia general de PINNs implementadas con optimizacion por etapas (Adam + L-BFGS) para PDEs de fractura/flujo en medios porosos, se usa el repositorio original de PINNs:

- **maziarraissi/PINNs** &mdash; https://github.com/maziarraissi/PINNs — implementacion de referencia de Raissi et al. (2019), la base metodologica que este paper y la mayoria de PINNs de este corpus citan.

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Parametros fisicos (Tabla 1 del paper: caso $E'=30$ GPa, $\mu=8.0$ Pa.s, $H=2.0$ m, $C_L=1.0$)

In [ ]:
E_prime = 30.0   # modulo elastico efectivo (normalizado, GPa)
mu = 8.0         # viscosidad del fluido [Pa.s]
H = 2.0          # altura de la fractura [m]
C_L = 1.0        # coeficiente de leak-off
t0 = 0.0         # tiempo de referencia
lambda_bc = 30.0  # peso de la perdida de contorno/inicial (Seccion 2.1.1)
coef = E_prime / (128 * mu * H)

## 2. Red PINN (Seccion 3.2.1: 4 capas ocultas x 256 neuronas, tanh) con salida no negativa

In [ ]:
class NordgrenPINN(nn.Module):
    def __init__(self, n_hidden=4, n_neurons=256):
        super().__init__()
        layers = [nn.Linear(2, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, 1)]
        self.net = nn.Sequential(*layers)
        self.softplus = nn.Softplus()

    def forward(self, xt):
        # Algoritmo 1, paso 4 (opcional en el paper): w0 = softplus(salida cruda) >= 0,
        # correccion que el propio paper senala como remedio a la subestimacion sistematica.
        return self.softplus(self.net(xt))


def d_d(f, v, idx):
    g = torch.autograd.grad(f, v, grad_outputs=torch.ones_like(f),
                             create_graph=True, retain_graph=True)[0]
    return g[:, idx:idx + 1]

## 3. Funcion de perdida (Eq. 1-4): residuo de Nordgren + contorno ($x=0,1$) + condicion inicial ($t=0$)

In [ ]:
N_col = 5000  # colocacion, Seccion 2.1.1
xt_col = torch.rand(N_col, 2, device=device)
xt_col.requires_grad_(True)

N_b = 200
t_b = torch.rand(N_b, 1)
xt_bc0 = torch.cat([torch.zeros(N_b, 1), t_b], dim=1).to(device).requires_grad_(True)   # x=0
xt_bc1 = torch.cat([torch.ones(N_b, 1), t_b], dim=1).to(device).requires_grad_(True)    # x=1
x_i = torch.rand(N_b, 1)
xt_init = torch.cat([x_i, torch.zeros(N_b, 1)], dim=1).to(device).requires_grad_(True)  # t=0


def physics_residual(model, xt):
    w = model(xt)
    w2 = w**2
    w2_x = d_d(w2, xt, 0)
    w2_xx = d_d(w2_x, xt, 0)
    w_t = d_d(w, xt, 1)
    t_eff = torch.clamp(xt[:, 1:2] - t0, min=1e-3)  # evita la singularidad 1/sqrt(t-t0) en t=t0
    leak_off = 8 * C_L / (np.pi * torch.sqrt(t_eff))
    return coef * w2_xx - leak_off - w_t


def compute_loss(model):
    res = physics_residual(model, xt_col)
    loss_physics = torch.mean(res**2)

    w_bc0 = model(xt_bc0)
    w_bc1 = model(xt_bc1)
    w_init = model(xt_init)
    loss_boundary = torch.mean(w_bc0**2) + torch.mean(w_bc1**2) + torch.mean(w_init**2)

    return loss_physics + lambda_bc * loss_boundary, loss_physics.item(), loss_boundary.item()

## 4. Entrenamiento por etapas: Adam (con decaimiento exponencial) seguido de L-BFGS (Seccion 2.1.1, Algoritmo 1)

In [ ]:
model = NordgrenPINN(n_hidden=4, n_neurons=256).to(device)

opt_adam = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(opt_adam, step_size=500, gamma=0.9)
history = []

for epoch in range(1000):
    opt_adam.zero_grad()
    loss, l_phy, l_bc = compute_loss(model)
    loss.backward()
    opt_adam.step()
    scheduler.step()
    history.append(loss.item())
    if epoch % 200 == 0:
        print(f'[Adam] epoch {epoch:5d} | loss={loss.item():.4e} | physics={l_phy:.4e} | boundary={l_bc:.4e}')

opt_lbfgs = torch.optim.LBFGS(model.parameters(), lr=0.5, max_iter=200,
                               history_size=30, line_search_fn='strong_wolfe')

def closure():
    opt_lbfgs.zero_grad()
    loss, _, _ = compute_loss(model)
    loss.backward()
    return loss

for step in range(10):
    loss = opt_lbfgs.step(closure)
    history.append(loss.item())
    print(f'[L-BFGS] step {step:3d} | loss={loss.item():.4e}')

## 5. Resultados: superficie $w_0(x,t)$ y evolucion en la boca de la fractura (cf. Fig. 1-3 del paper)

In [ ]:
n_side = 60
xs = np.linspace(0, 1, n_side)
ts = np.linspace(0, 1, n_side)
X, Tt = np.meshgrid(xs, ts)
xt_grid = torch.tensor(np.stack([X.ravel(), Tt.ravel()], axis=1), dtype=torch.float32, device=device)
with torch.no_grad():
    w_grid = model(xt_grid).cpu().numpy().reshape(n_side, n_side)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
im = axes[0].contourf(X, Tt, w_grid, levels=30, cmap='viridis')
axes[0].set_xlabel('x'); axes[0].set_ylabel('t')
axes[0].set_title('$w_0(x,t)$ predicho por la PINN')
plt.colorbar(im, ax=axes[0])

axes[1].plot(xs, w_grid[-1, :])
axes[1].set_xlabel('x'); axes[1].set_ylabel('$w_0(x, t=1)$')
axes[1].set_title('Ancho de fractura en t=1.0')
axes[1].grid(alpha=0.3)

axes[2].semilogy(history)
axes[2].axvline(1000, color='gray', linestyle=':', label='inicio L-BFGS')
axes[2].set_xlabel('Iteracion'); axes[2].set_ylabel('Loss (escala log)')
axes[2].set_title('Convergencia (Adam -> L-BFGS)')
axes[2].legend()
plt.tight_layout()
plt.show()

Nota: el paper (Seccion 5.1.1) reporta que, incluso con optimizadores combinados, la PINN tiende a **subestimar** el crecimiento cuadratico esperado del ancho de fractura, y sugiere como remedios la no-dimensionalizacion, mayor densidad de puntos de frontera, ponderacion adaptativa de la perdida y (como se aplica aqui) forzar $w_0\geq0$. Para la parte de optimizacion cuantica (VQE) del paper, que resuelve una version discretizada linealizada del mismo problema como un Hamiltoniano en Qiskit, ver el paper original (Seccion 4).